# Lab 9: Planning and comparing groups with stop-signal contrasts

**Synthetic dataset v1.0.1**

**Overall question:** Does the modeled correct-stop-minus-correct-go contrast in right pars opercularis differ across reported substance-use frequency groups in this synthetic cohort, and does the conclusion hold under a stricter imaging-quality rule?

**Learning objectives.** By the end of this lab, you should be able to:

- explain what the right pars opercularis correct-stop-minus-correct-go contrast measures, and distinguish it from SSRT;
- separate imaging availability, primary eligibility, substance-report resolution, and stricter sensitivity eligibility;
- plan a two-group comparison's sample size from a stated difference, variability, α, and target power;
- run and interpret a one-way ANOVA F-test without assuming it identifies a particular pair;
- apply a Bonferroni adjustment to three planned pairwise comparisons and interpret their estimates and intervals;
- compare primary and stricter-quality results without choosing a sample after seeing significance.

**How this lab connects.** In Exercise 9 your team planned this comparison in a study brief (sections 1–6), including a provisional difference worth detecting. Today's warm-up practiced power and the difference between an overall test and specific pairs. Lab 8 compared two groups with Welch's method; here three groups share one ANOVA model.

Complete **three Your Turns**: power, ANOVA, and the Bonferroni pairs. Supplied code loads and checks the data, demonstrates each method, and draws plots. Interpretation prompts prepare you for the **Lab 9 Canvas reflection**. Discuss or think through them as you work; written notebook notes are optional. Write the assessed responses once, in Canvas. No notebook upload is required.

The records are synthetic, not real ABCD participant data. No personal substance-use disclosure is requested. Reading: Vu and Harrington §§5.4–5.6.

Use Copilot for hints or debugging after your attempt. Complete Canvas independently without generative AI answering, drafting, rewriting, or checking responses. Approved accessibility, spelling/grammar, and instructor-authorized translation tools remain permitted.


## Section 1. Plan and audit the analysis sample

Before looking at group results, set **CHOICE** (cannabis or alcohol) and **DETAIL_PAIR** (the pair you will explain in detail). Keep **MAGNITUDE_PSC** at **0.0266 PSC**, or use your planning target from Exercise 9 if you have it. Keep all three settings for the whole lab.

The planned pairwise tests are **two-sided**. The overall ANOVA F-test asks whether the three population means are all equal and uses the upper tail of the F distribution, at **α = .05**. The three planned pairs form one family with **α = .05/3 per pair** (Bonferroni). You will report all three pairs whatever the F-test shows. Do not change substance, magnitude, pairs, or exclusions to obtain a preferred result.

Run the supplied setup. It loads one synthetic wave, checks the file, and records the analysis plan.


In [ ]:
from pathlib import Path
import hashlib, json, math, os, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
try:
    from IPython.display import display, Markdown
except ImportError:  # Plain-Python review; Jupyter uses its normal rich display.
    def display(*objects):
        for obj in objects:
            print(obj)
    Markdown = str

CHOICE = 'cannabis'     # Set 'cannabis' or 'alcohol' BEFORE viewing group results.
MAGNITUDE_PSC = 0.0266  # Keep the default or use your Exercise 9 target (0.0133, 0.0266, or 0.0399 PSC).
DETAIL_PAIR = (2, 0)    # Choose now: (1, 0) lower − zero, (2, 0) higher − zero, or (2, 1) higher − lower.
assert CHOICE in ('cannabis', 'alcohol'), "Set CHOICE to 'cannabis' or 'alcohol'."
assert MAGNITUDE_PSC in (0.0133, 0.0266, 0.0399), 'Set MAGNITUDE_PSC to one of the three Exercise 9 planning values.'
ALPHA = 0.05            # overall F-test; Bonferroni family alpha for the three pairs
FAMILY_SIZE = 3
REFERENCE_SD = 0.133    # PSC; supplied calibration summary, not fitted to any group result
PAIRS = [(1, 0), (2, 0), (2, 1)]  # lower - zero, higher - zero, higher - lower
assert DETAIL_PAIR in PAIRS, 'Set DETAIL_PAIR to (1, 0), (2, 0), or (2, 1).'
NOTEBOOK_VERSION = '1.1'
DATASET_VERSION = '1.0.1'
# Fingerprint of the v1.0.1 CSV with line endings normalized, so the check passes
# whether the file was stored with Windows or Unix line endings.
EXPECTED_SHA256 = 'ebb65a6b98e834f97d006ec175a855905f8c3bfc0f74befff240fd08a8b96899'
# Optional override: a folder containing M8_M9_STUDENT.csv and its documentation.
DATA_DIR_OVERRIDE = None
if DATA_DIR_OVERRIDE is not None:
    candidates = [Path(DATA_DIR_OVERRIDE)]
else:
    candidates = [
        path
        for base in (Path.cwd(), *Path.cwd().parents)
        for path in (base / 'data' / 'labs_08_09', base / 'release_v1_0_1' / 'student')
    ]
DATA = next((p for p in candidates if (p / 'M8_M9_STUDENT.csv').is_file()), None)
if DATA is None:
    raise FileNotFoundError('Cannot find M8_M9_STUDENT.csv in data/labs_08_09 or release_v1_0_1/student. '
                            'Open this notebook from the labs/ folder of the course repository, or set DATA_DIR_OVERRIDE.')
csv_path = DATA / 'M8_M9_STUDENT.csv'
csv_sha256 = hashlib.sha256(csv_path.read_bytes().replace(b'\r\n', b'\n')).hexdigest()
assert csv_sha256 == EXPECTED_SHA256, ('This CSV differs from the v1.0.1 release. Restore the original '
                                       'file from the repository; do not edit it.')
df = pd.read_csv(csv_path, dtype={'participant_id': str, 'instructional_wave_id': str})
assert len(df) == 5000 and df.participant_id.is_unique and df.instructional_wave_id.nunique() == 1
assert df.instructional_tlfb_window_days.eq(365).all()

OUT = 'mr_y_tfmri__sst__csvcg__dsk__pop__rh_beta'
GROUP = f'instructional_{CHOICE}_frequency_group'
PRIMARY = f'instructional_m9_{CHOICE}_analysis_eligible'
REASON = f'instructional_m9_{CHOICE}_eligibility_reason'
RESOLVED = f'instructional_{CHOICE}_report_resolved'
STRICT = 'instructional_sst_sensitivity_eligible'
OTHER = 'alcohol' if CHOICE == 'cannabis' else 'cannabis'
OTHER_ANY = f'instructional_{OTHER}_any_detailed'
CUTS = {'cannabis': ('0 days', '1–11 days', '12+ days'), 'alcohol': ('0 days', '1–5 days', '6+ days')}[CHOICE]
LABELS = dict(zip((0, 1, 2), CUTS))
PAIR_NAMES = {p: f'{LABELS[p[0]]} − {LABELS[p[1]]}' for p in PAIRS}
pd.set_option('display.precision', 5)

analysis_plan = pd.DataFrame([
    ['Outcome', 'Right pars opercularis, correct stop − correct go (PSC), one value per person'],
    ['Groups', f'{CHOICE}: ' + ' / '.join(CUTS) + ' in the fixed 365-day period'],
    ['Primary sample', f'Primary stop-signal eligibility and a resolved {CHOICE} report'],
    ['Stricter sample', 'Stricter imaging quality and a resolved report; same outcome, groups, pairs, and family'],
    ['Overall test', 'One-way ANOVA F-test, alpha = .05'],
    ['Planned pairs', '; '.join(PAIR_NAMES.values()) + ' (all reported)'],
    ['Pairwise method', 'ANOVA-pooled MSE, error df = N − 3; Bonferroni alpha .05/3 per pair; two-sided'],
    ['Difference worth detecting', f'{MAGNITUDE_PSC} PSC (d = {MAGNITUDE_PSC / REFERENCE_SD:.2f} with reference SD {REFERENCE_SD} PSC)'],
    ['Pair explained in detail', PAIR_NAMES[DETAIL_PAIR]],
], columns=['item', 'plan'])
display(analysis_plan)
print('Choice:', CHOICE, '| Dataset:', DATASET_VERSION, '| Documentation folder:', DATA)
for filename in ['README.md', 'M8_M9_DATA_DICTIONARY.md', 'M8_M9_ANALYSIS_GUIDE.md']:
    assert (DATA / filename).is_file(), f'Missing student documentation: {filename}'
    relative = Path(os.path.relpath(DATA / filename, Path.cwd())).as_posix().replace(' ', '%20')
    display(Markdown(f'[{filename}]({relative})'))


### The measure

In the **stop-signal task (SST)**, people respond quickly to a go signal and try to withhold the response when a stop signal follows. The outcome is a modeled regional contrast in **right pars opercularis**, part of the right inferior frontal region: **correct stop minus correct go**, in percent signal change (PSC). Each eligible person contributes one value.

- **Not a pure inhibition score.** Correct-stop and correct-go trials differ in more than stopping: stop trials add a second signal and end without a response.
- **Higher is not automatically better.** A larger contrast does not by itself mean better control.
- **Not SSRT.** The stop-signal reaction time (SSRT, in milliseconds) is a behavioral estimate from the same task. It is not this lab's outcome and is not in the release.
- **Not the whole network.** One right-hemisphere region is not the whole stopping network.

The groups are the release's fixed frequency groups for your substance in the synthetic 365-day period: cannabis 0 / 1–11 / 12+ days, alcohol 0 / 1–5 / 6+ days. The labels describe reported frequency, not severity. Same-wave reports establish neither temporal order nor causation.


### See the region behind the number

`dsk` in the field name refers to the **Desikan atlas**, which divides each person's cortical surface into labeled regions. Pars opercularis is one of those regions. This is different from Lab 8's nucleus accumbens, a subcortical structure outlined by `aseg`. The value is the modeled response averaged across the voxels in that region: one number per person.

![DEAP brain image for mr_y_tfmri__sst__csvcg__dsk__pop__rh_beta: a lateral view of the right hemisphere's cortical surface with pars opercularis highlighted in blue, toward the front of the brain above the lateral fissure.](../assets/l9_deap_sst_csvcg_pop_rh_roi.png)

*Source: DEAP data dictionary screenshot supplied by the instructor, November 2026. DEAP describes the field as an average beta weight for the Stop Signal task (correct stop vs. correct go; all runs) in the Desikan region pars opercularis, right hemisphere. The records in this lab are synthetic values that reuse this field's name and definition; they are not ABCD participant data.*

**Optional:** in the [DEAP data dictionary](https://abcd.deapscience.com/#/my-datasets/create-dataset), search for `mr_y_tfmri__sst__csvcg__dsk__pop__rh_beta` and open its brain image. It is interactive: rotate it to see where the region sits. DEAP may ask you to sign in or may be unavailable; the screenshot above is all you need.

For the Canvas reflection, Question 1 uses this measure description.


### Review the lab’s analysis plan

In Exercise 9, your team proposed a research question, an outcome, a planning target, and an imaging-quality policy. This lab supplies its own analysis plan in the table below. Its primary and stricter samples follow fixed eligibility rules, regardless of the policy your team preferred.

**You do not need your completed Exercise 9 worksheet.** If you have it, you may compare your responses in Questions 6–8 with the table. Otherwise, use the supplied plan. Comparison notes are optional.

For `MAGNITUDE_PSC` in the setup cell above, you may carry forward your team’s planning target. If you do not have it, use the supplied default of **0.0266 PSC**. Set this before viewing group results and keep it unchanged throughout the lab.

In [ ]:
brief_to_release_check = pd.DataFrame([
    [1, 'Research question', f'Does the mean SST contrast differ across the three {CHOICE} frequency groups among M9-eligible participants?'],
    [3, 'Comparison and analysis unit', 'One contrast per person; three independent groups; pairs: ' + '; '.join(PAIR_NAMES.values())],
    [4, 'Who is included', f'Release primary rule: {PRIMARY} == 1 (primary SST eligibility + resolved {CHOICE} report)'],
    [5, 'Sensitivity rule', 'Release stricter SST quality subset + resolved report; outcome, cutoffs, pairs, and family unchanged'],
    [6, 'Difference worth detecting', f'{MAGNITUDE_PSC} PSC (d = {MAGNITUDE_PSC / REFERENCE_SD:.2f}); two-sided; alpha .05/3 per pair'],
], columns=['brief_section', 'heading', 'release_rule_implemented'])
# Optional notes for your own use: type your team's entry and any difference.
brief_to_release_check['your_brief_entry'] = ''
brief_to_release_check['difference_and_reason'] = ''
display(brief_to_release_check)


### Who enters the analysis

Records pass through separate steps: imaging must be **available**, the task must pass **primary quality**, and your substance report must be **resolved** (a blank report is not zero). Together these define the **primary sample**. The **stricter sample** additionally requires a stricter task-quality rule; it is a subset of the primary sample. Unresolved reports are not assigned to a group.

**Predict before running:** which step removes the most records? Roughly what share of the primary sample will the stricter rule remove? Then check against the audit. The audit shows counts only; it shows no group outcome means.


In [ ]:
sst_ok = df.instructional_sst_analysis_eligible.eq(1)
resolved = df[RESOLVED].eq(1)
assert df[PRIMARY].eq(sst_ok & resolved).all(), 'Primary flag should equal SST eligibility plus a resolved report.'
primary = df.loc[df[PRIMARY].eq(1)].copy()
stricter = df.loc[df[STRICT].eq(1) & resolved].copy()
assert set(stricter.participant_id) <= set(primary.participant_id), 'The stricter sample must be a subset of the primary sample.'
assert primary[[OUT, GROUP]].notna().all().all(), 'Every primary record needs an outcome and a group.'
# Rebuilt from the data on every run, so re-running this cell is safe.
eligibility_audit = pd.DataFrame([
    ['Starting records', len(df)],
    ['SST imaging unavailable', int(df.instructional_sst_eligibility_reason.eq('imaging_unavailable').sum())],
    ['SST failed primary quality', int(df.instructional_sst_eligibility_reason.eq('failed_primary_quality').sum())],
    ['SST primary eligible', int(sst_ok.sum())],
    [f'SST eligible but {CHOICE} report unresolved', int((sst_ok & ~resolved).sum())],
    [f'Primary sample ({CHOICE})', len(primary)],
    ['Primary but fails stricter quality', int((df[PRIMARY].eq(1) & df[STRICT].eq(0)).sum())],
    [f'Stricter sample ({CHOICE})', len(stricter)],
], columns=['stage', 'n'])
assert eligibility_audit.n.iloc[1] + eligibility_audit.n.iloc[2] + eligibility_audit.n.iloc[3] == len(df)
display(eligibility_audit)
counts = {name: s[GROUP].value_counts().reindex([0, 1, 2]).astype(int) for name, s in (('primary', primary), ('stricter', stricter))}
allocation_table = pd.DataFrame({
    'group': [LABELS[k] for k in (0, 1, 2)],
    'n_primary': counts['primary'].values, 'share_primary': (counts['primary'] / counts['primary'].sum()).values,
    'n_stricter': counts['stricter'].values, 'share_stricter': (counts['stricter'] / counts['stricter'].sum()).values})
display(allocation_table)


### Exclusions are research decisions

- **Report the steps.** State how many records were excluded at each step and why.
- **Ask who is missing.** Excluded people may differ from those analyzed, because scan quality can relate to characteristics such as age or attention. The analyzed groups may not represent everyone who was scanned.
- **Stricter quality has limits.** A stricter quality rule cannot repair task-design limits, such as the stimulus differences built into the stop − go contrast.

For the Canvas reflection, Question 2 asks you to apply the release eligibility rules to an invented record.


## Section 2. How many people would we need to detect our planned difference?

In Lab 8 you read intervals against ±0.02 PSC, a value stipulated for teaching. In Exercise 9 your team chose a provisional magnitude for this SST contrast (0.0133, 0.0266, or 0.0399 PSC) and wrote its rationale and limits. Both are teaching choices, not established thresholds, and they concern different regions and tasks. Here is how such a value enters a study plan.

**Standardized difference.** d = delta / SD, where delta is the difference in PSC and SD is a reference spread of person-to-person values. This lab uses a reference SD of **0.133 PSC** from a supplied calibration summary, not from any group result. So 0.0266 PSC corresponds to d = 0.20.

**Required n per group** (Vu and Harrington §5.4.3): for equal groups, a common SD, and a two-sided test, the normal approximation is

n per group = ceil( 2 × (z₁₋α/₂ + z_power)² / d² ).

This gives n **per group**, not the total, and it is an approximation, not an exact t-based calculation.

**Which α?** A standalone comparison uses α = .05. Each of our three planned pairs belongs to a declared family, so each uses α = .05/3. A smaller α guards against false positives (Type I errors) but needs more people for the same power (more Type II risk at a fixed n).

**Power** is the probability of rejecting H₀ when the specified difference is real, under the assumptions. It is not the probability that H₀ is false. It is planned **before** results and is never computed from an observed difference.


In [ ]:
def required_n(d, alpha, power=0.80):
    # Equal groups, common SD, two-sided normal approximation (Vu and Harrington §5.4.3); n per group.
    z_a = stats.norm.ppf(1 - alpha / 2)
    z_b = stats.norm.ppf(power)
    return math.ceil(2 * (z_a + z_b) ** 2 / d ** 2)

def approx_power(d, n_a, n_b, alpha):
    # Two-sided normal-approximation power for two groups of sizes n_a and n_b.
    z = d / math.sqrt(1 / n_a + 1 / n_b)
    c = stats.norm.ppf(1 - alpha / 2)
    return stats.norm.cdf(z - c) + stats.norm.cdf(-z - c)

power_scenarios = pd.DataFrame([dict(delta_PSC=v, d=round(v / REFERENCE_SD, 2),
                                     n_per_group_alpha_05=required_n(v / REFERENCE_SD, ALPHA),
                                     n_per_group_alpha_05_over_3=required_n(v / REFERENCE_SD, ALPHA / FAMILY_SIZE))
                                for v in (0.0133, 0.0266, 0.0399)])
power_scenarios['your_choice'] = power_scenarios.delta_PSC.eq(MAGNITUDE_PSC)
display(power_scenarios)
D_PLAN = MAGNITUDE_PSC / REFERENCE_SD
power_plan = pd.Series({'substance': CHOICE, 'delta_PSC': MAGNITUDE_PSC, 'reference_SD_PSC': REFERENCE_SD,
                        'd': round(D_PLAN, 3), 'alpha_per_pair': round(ALPHA / FAMILY_SIZE, 5), 'target_power': 0.80,
                        'n_per_group_needed': required_n(D_PLAN, ALPHA / FAMILY_SIZE), 'test': 'two-sided',
                        'basis': 'equal groups, common SD, normal approximation'}).to_frame('plan')
display(power_plan)
ns = np.arange(20, 2501, 10)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for v, style in zip((0.0133, 0.0266, 0.0399), ('-', '--', ':')):
    d = v / REFERENCE_SD
    axes[0].plot(ns, [approx_power(d, k, k, ALPHA / FAMILY_SIZE) for k in ns], style, color='#13678A', label=f'delta {v} PSC (d = {d:.1f})')
for a, style, name in ((ALPHA, '-', 'alpha .05 (standalone)'), (ALPHA / FAMILY_SIZE, '--', 'alpha .05/3 (one of three pairs)')):
    axes[1].plot(ns, [approx_power(D_PLAN, k, k, a) for k in ns], style, color='#B34B14', label=name)
for ax, title in ((axes[0], 'Change the difference; alpha .05/3 fixed'), (axes[1], f'Change alpha; delta {MAGNITUDE_PSC} PSC fixed')):
    ax.axhline(0.8, color='black', linewidth=0.8, linestyle='-.')
    ax.set(xlabel='n per group (equal groups)', ylabel='Planning power', ylim=(0, 1.02), title=title)
    ax.legend(fontsize=8, loc='lower right')
fig.suptitle('Teaching plan: normal approximation, reference SD 0.133 PSC; dash-dot line marks power 0.80')
power_display = fig
fig.tight_layout(); plt.show()


### Your Turn 1: change one input and recompute n

Your plan uses delta = `MAGNITUDE_PSC`, SD = 0.133 PSC, α = .05/3, and power = 0.80. In the next cell, change **exactly one** of `yt1_delta`, `yt1_sd`, `yt1_alpha`, or `yt1_power`. Then compute `yt1_n` with the formula above: d = delta / SD, the z values from `stats.norm.ppf`, and `math.ceil` to round up.

**Predict first:** will your change raise or lower the required n, and why?

Optional Copilot prompt after trying: "Explain how each input in a two-group sample-size formula changes n, using a different invented example. Give one hint at a time, not my code."


In [ ]:
# Your Turn 1: change exactly one input, then compute n per group.
yt1_delta, yt1_sd, yt1_alpha, yt1_power = MAGNITUDE_PSC, REFERENCE_SD, ALPHA / FAMILY_SIZE, 0.80
yt1_n = None  # math.ceil( ... )


In [ ]:
assert yt1_n is not None, 'Compute yt1_n with the formula.'
baseline = dict(delta=MAGNITUDE_PSC, sd=REFERENCE_SD, alpha=ALPHA / FAMILY_SIZE, power=0.80)
mine = dict(delta=yt1_delta, sd=yt1_sd, alpha=yt1_alpha, power=yt1_power)
changed = [k for k in baseline if not math.isclose(baseline[k], mine[k])]
assert len(changed) == 1, f'Change exactly one input; you changed {len(changed)}: {changed}.'
assert yt1_delta > 0 and yt1_sd > 0 and 0 < yt1_alpha < 1 and 0 < yt1_power < 1, 'Use a positive delta and SD, and alpha and power between 0 and 1.'
expected_n = required_n(yt1_delta / yt1_sd, yt1_alpha, yt1_power)
assert yt1_n == expected_n, (f'Expected {expected_n}. Check d = delta / SD, the z value for 1 − alpha/2 '
                             '(two-sided), the z value for the power, and rounding up with math.ceil.')
power_change = pd.DataFrame([dict(version='your plan', **baseline, n_per_group=required_n(D_PLAN, ALPHA / FAMILY_SIZE)),
                             dict(version=f'changed {changed[0]}', **mine, n_per_group=yt1_n)])
display(power_change)


### Unequal groups: planning power for each planned pair

Real groups are rarely equal. Each planned pair compares two groups whose **eligible counts** come from Section 1; no group outcomes are used. With a common SD, the SE of a pair's difference is SD × √(1/nᵢ + 1/nⱼ). The **balanced-equivalent size** n_equiv = 2 / (1/nᵢ + 1/nⱼ) is the equal-group size with the same SE. It is not the total enrollment. A pair that involves a small group has less power than the total cohort size suggests.

The supplied cell gives the planning power of **each of the three planned pairs** for your difference (`MAGNITUDE_PSC`), at α = .05/3 and α = .05. Your `DETAIL_PAIR` is marked. These numbers describe the design, not the observed result. Each is the power of a **two-group** comparison; none is the power of the overall ANOVA F-test.


In [ ]:
pair_counts = counts['primary']
allocation_plan = pd.DataFrame([dict(comparison=PAIR_NAMES[(i, j)], n_first=int(pair_counts[i]), n_second=int(pair_counts[j]),
                                     balanced_equivalent_n=round(2 / (1 / pair_counts[i] + 1 / pair_counts[j]), 1),
                                     power_alpha_05_over_3=approx_power(D_PLAN, pair_counts[i], pair_counts[j], ALPHA / FAMILY_SIZE),
                                     power_alpha_05=approx_power(D_PLAN, pair_counts[i], pair_counts[j], ALPHA),
                                     detail_pair=(i, j) == DETAIL_PAIR)
                                for i, j in PAIRS])
print(f'Planning power for delta = {MAGNITUDE_PSC} PSC (d = {D_PLAN:.2f}); two-group calculations, not ANOVA power')
display(allocation_plan)


### Interpret and prepare for Canvas

- What difference did your plan target, in PSC and as d, and what does that value not establish?
- In each panel of `power_display`, which single input changed, and how did power respond?
- Why does α = .05/3 need more people than α = .05 for the same power? Which error does each choice guard against?
- Which planned pair has the lowest planning power, and why? Is the whole cohort "well powered"? Answer only for a stated difference, pair, and α.

**Have ready:** locate your planned difference in PSC and as d and the planning power of your detail pair. Be ready to explain the connection to Type II error; no written notebook response is required.

For the Canvas reflection, Question 3 asks how to interpret a planned difference. Question 4 asks how the stricter per-pair threshold changes required sample size. See `power_plan` and `power_scenarios`.


## Section 3. Do the three group means differ at all?

**The question.** Is there evidence that the three population means are not all equal? One-way ANOVA compares two kinds of variation:

- **between groups:** how far the group means sit from the overall mean;
- **within groups:** how far people sit from their own group's mean.

**The pieces.** **MSE** (mean squared error) is the within-group variance pooled across all three groups, with error df = N − 3. **F** = (between-group mean square) / MSE. A large F means group means differ more than within-group variation would usually produce. A significant F says **not all** means are equal; it does not say which pair differs.

**Assumptions.**
- Independent people.
- Roughly normal values or large groups, with attention to tails.
- A **common within-group variance**. This differs from Lab 8's Welch comparison, which allowed unequal variances. A similar ratio of group SDs is descriptive support, not proof.

### Demo: a constructed A/B/C example

The next cell builds a small **constructed teaching dataset**: 10 invented values per group, with fixed residuals so each group has SD 0.10. It is designed to show a rejected F-test and a mix of pairwise results. It is not study data and has nothing to do with either substance.


In [ ]:
# Constructed teaching data, not study data: fixed residuals scaled to SD 0.10; 10 values per group.
base = np.array([-1.5, -1.0, -0.7, -0.3, -0.1, 0.1, 0.3, 0.7, 1.0, 1.5])
resid = (base - base.mean()) / base.std(ddof=1) * 0.10
demo = pd.DataFrame([dict(demo_id=f'{g}{k + 1:02d}', group=g, value=mu + r)
                     for g, mu in (('A', 0.05), ('B', 0.15), ('C', 0.21)) for k, r in enumerate(resid)])
demo_group_summary = demo.groupby('group')['value'].agg(n='count', mean='mean', SD='std')
demo_N = len(demo)
demo_grand = demo.value.mean()
demo_ss_between = float((demo_group_summary.n * (demo_group_summary['mean'] - demo_grand) ** 2).sum())
demo_ss_within = float(((demo_group_summary.n - 1) * demo_group_summary.SD ** 2).sum())
demo_mse = demo_ss_within / (demo_N - 3)
demo_F = (demo_ss_between / 2) / demo_mse
demo_p = stats.f.sf(demo_F, 2, demo_N - 3)
assert np.isclose(demo_F, stats.f_oneway(*[demo.loc[demo.group.eq(g), 'value'] for g in 'ABC']).statistic)
demo_anova_table = pd.DataFrame([
    dict(source='Between groups', SS=demo_ss_between, df=2, MS=demo_ss_between / 2, F=demo_F, p=demo_p),
    dict(source='Within groups (error)', SS=demo_ss_within, df=demo_N - 3, MS=demo_mse, F=np.nan, p=np.nan)])
print('Constructed teaching example (not study data):')
display(demo_group_summary, demo_anova_table)


### Your Turn 2: summarize your groups and run the F-test

Using the `primary` sample, the `GROUP` column, and the `OUT` column:

1. create `group_summary`, one row per group (0, 1, 2), with columns `n`, `mean`, and `SD`, using `groupby` and `agg`, as in the demo;
2. run `stats.f_oneway` on the three groups' outcome values, in the order 0, 1, 2, and save the result as `anova_result`.

**Predict first:** given your Section 2 plan, what would a nonsignificant F mean, and what would it not mean?


In [ ]:
# Your Turn 2
group_summary = None  # primary.groupby(...)[...].agg(n=..., mean=..., SD=...)
anova_result = None  # stats.f_oneway( ... )


In [ ]:
assert group_summary is not None and anova_result is not None, 'Complete both lines.'
assert list(group_summary.columns) == ['n', 'mean', 'SD'], 'Name the summary columns n, mean, and SD, in that order.'
assert [int(k) for k in group_summary.index] == [0, 1, 2], 'Group the primary sample by GROUP so the rows are 0, 1, 2.'
check = primary.groupby(GROUP)[OUT]
assert (group_summary['n'].values == check.count().values).all(), 'Use the primary sample: the counts do not match.'
assert np.allclose(group_summary['mean'], check.mean()), 'Summarize the OUT column of the primary sample.'
assert np.allclose(group_summary['SD'], check.std(ddof=1)), "Use the sample SD ('std', which divides by n − 1)."
N = int(group_summary.n.sum())
df_between, df_error = 2, N - 3
mse = float(((group_summary.n - 1) * group_summary.SD ** 2).sum() / df_error)
grand = float(primary[OUT].mean())
ss_between = float((group_summary.n * (group_summary['mean'] - grand) ** 2).sum())
F_check = (ss_between / df_between) / mse
assert np.isclose(anova_result.statistic, F_check), 'Pass the three groups (0, 1, 2) of the primary sample to stats.f_oneway.'
anova_table = pd.DataFrame([
    dict(source='Between groups', SS=ss_between, df=df_between, MS=ss_between / df_between, F=anova_result.statistic, p=anova_result.pvalue),
    dict(source='Within groups (error)', SS=mse * df_error, df=df_error, MS=mse, F=np.nan, p=np.nan)])
display(group_summary.rename(index=LABELS), anova_table)
group_diagnostic = primary.groupby(GROUP)[OUT].describe(percentiles=[.01, .5, .99]).rename(index=LABELS)
display(group_diagnostic)
print('Largest / smallest group SD:', round(group_summary.SD.max() / group_summary.SD.min(), 3), '(descriptive, not a test)')
context_rows = []
for name, sample in (('primary', primary), ('stricter', stricter)):
    for k in (0, 1, 2):
        records = sample.loc[sample[GROUP].eq(k)]
        observed = records[OTHER_ANY].notna()
        positive = int(records.loc[observed, OTHER_ANY].eq(1).sum())
        context_rows.append(dict(sample=name, group=LABELS[k], n=len(records), other_observed_n=int(observed.sum()),
                                 other_positive_n=positive, other_positive_percent=100 * positive / observed.sum() if observed.sum() else np.nan,
                                 mean_age=records.ab_g_dyn__visit_age.mean()))
context_summary = pd.DataFrame(context_rows)
print('Other substance:', OTHER, '(co-use percentages use observed other-substance reports as the denominator)')
display(context_summary)
fig, ax = plt.subplots(figsize=(7, 3.8))
values = [primary.loc[primary[GROUP].eq(k), OUT] for k in (0, 1, 2)]
ax.boxplot(values, positions=[0, 1, 2], showfliers=True)
ax.scatter([0, 1, 2], group_summary['mean'], marker='D', color='#B34B14', zorder=3, label='Group mean')
ax.set(xticks=[0, 1, 2], xticklabels=[f'{LABELS[k]}\nn = {int(group_summary.n.iloc[k])}' for k in (0, 1, 2)],
       ylabel='Correct stop − correct go (PSC)', title=f'{CHOICE.title()} groups, primary sample: spread, tails, and means')
ax.legend(loc='upper right', fontsize=8)
group_plot = fig
fig.tight_layout(); plt.show()


### Interpret and prepare for Canvas

- Report F, both df, and p. What does the overall test conclude at α = .05, and what can it not tell you about specific pairs?
- If F is not significant, does that show the three population means are equal? Section 2 planned power for two-group comparisons, not for this overall test. The pairwise intervals in Section 4 show what remains uncertain.
- Check the assumptions: group sizes (unequal n is allowed), the SD ratio, and the tails in `group_diagnostic` and `group_plot`. Which one would most concern you, and why?
- `context_summary` describes co-use and age by group. It does not adjust the comparison; the groups were not randomly assigned.

**Have ready:** locate F, both df, and p, and choose one assumption check. In Canvas, Question 5 asks for one check with supporting evidence (1–2 sentences). Question 6 asks for the omnibus result, conclusion and limit (1–2 sentences). Notebook notes are optional.

For the Canvas reflection, Question 5 uses `group_summary`, `group_diagnostic`, or `group_plot`. Question 6 uses `anova_table`.


## Section 4. Which groups differ, and by how much?

**From Welch to MSE.** In Lab 8, Welch's method compared two groups without assuming equal variances. Here the three pairs come from one ANOVA model, which assumes a common within-group variance. Each pair therefore uses the **same MSE from all three groups** and the same error df = N − 3. The larger df comes from that model; it is not a reason to assume equal variance. This is not a separately pooled two-group t-test and not Welch.

For pair i − j: SE = √( MSE × (1/nᵢ + 1/nⱼ) ), and t = (meanᵢ − meanⱼ) / SE. The two-sided p-value uses df = N − 3.

**The family.** We declared three pairs before results:
- lower − zero;
- higher − zero;
- higher − lower.

To keep the chance of any false positive in the family at or below .05, each pair uses α = .05/3, or equivalently **adjusted p = min(3 × raw p, 1)** compared with .05. Each interval uses t at 1 − .05/6, so its individual confidence level is 1 − .05/3, about **98.333%**. These are not ordinary 95% intervals. Together, the three intervals give at least 95% confidence for the family, within this sample. It is not a joint 95% guarantee across all six intervals.

**A deliberate departure.** Vu and Harrington §5.5.3 introduce pairwise follow-up after a rejected F-test. Here the three pairs were planned in advance, so we report all three **whatever the F-test shows**. That does not license hunting for a promising pair, or treating a nonsignificant F as evidence of a difference.

### Demo: one pair by hand, then all three

The supplied helper `pooled_pair` returns one pair's estimate, SE, t, df, raw p, and family-adjusted interval. The demo works B − A by hand first.


In [ ]:
def pooled_pair(summary, i, j, mse, df_error, alpha=ALPHA, family_size=FAMILY_SIZE):
    # ANOVA-pooled comparison of group i minus group j (Vu and Harrington §5.5.3), Bonferroni interval.
    estimate = summary.loc[i, 'mean'] - summary.loc[j, 'mean']
    se = math.sqrt(mse * (1 / summary.loc[i, 'n'] + 1 / summary.loc[j, 'n']))
    t = estimate / se
    raw_p = 2 * stats.t.sf(abs(t), df_error)
    t_crit = stats.t.ppf(1 - alpha / (2 * family_size), df_error)
    return dict(pair=(i, j), estimate=estimate, SE=se, t=t, df=df_error, raw_p=raw_p,
                low=estimate - t_crit * se, high=estimate + t_crit * se)

print('Constructed teaching example (not study data): B − A by hand')
b_minus_a = demo_group_summary.loc['B', 'mean'] - demo_group_summary.loc['A', 'mean']
se_ba = math.sqrt(demo_mse * (1 / 10 + 1 / 10))
t_ba = b_minus_a / se_ba
p_ba = 2 * stats.t.sf(abs(t_ba), demo_N - 3)
print(f'difference = {b_minus_a:.4f}; SE = sqrt({demo_mse:.4f} × (1/10 + 1/10)) = {se_ba:.4f}; '
      f't = {t_ba:.3f} on {demo_N - 3} df; raw p = {p_ba:.4f}; adjusted p = {min(3 * p_ba, 1):.4f}')
demo_pairwise_table = pd.DataFrame([pooled_pair(demo_group_summary, i, j, demo_mse, demo_N - 3)
                                    for i, j in (('B', 'A'), ('C', 'A'), ('C', 'B'))])
demo_pairwise_table['adjusted_p'] = np.minimum(FAMILY_SIZE * demo_pairwise_table.raw_p, 1)
display(demo_pairwise_table)
print('B − A: raw p < .05 but adjusted p > .05. C − A survives the adjustment. The F-test rejected, yet one pair does not differ clearly.')


### Your Turn 3: run the planned family

1. Set `family_alpha` to the per-pair α for a family of `FAMILY_SIZE` pairs at overall α = `ALPHA`.
2. Build `pairwise_table` as a DataFrame of `pooled_pair(group_summary, i, j, mse, df_error)` for each `(i, j)` in `PAIRS`, in that order. `mse` and `df_error` come from Section 3.
3. Add a column `adjusted_p` = min(3 × raw p, 1). `np.minimum` works on a whole column.

**Predict first:** before running, which interval do you expect to be widest, and why? Hint: look at the group sizes.


In [ ]:
# Your Turn 3
family_alpha = None  # per-pair alpha
pairwise_table = None  # pd.DataFrame([pooled_pair(...) for i, j in PAIRS])
# pairwise_table['adjusted_p'] = ...


In [ ]:
assert family_alpha is not None and pairwise_table is not None, 'Complete family_alpha and pairwise_table.'
assert math.isclose(family_alpha, ALPHA / FAMILY_SIZE), 'The per-pair alpha is the overall alpha divided by the number of planned pairs.'
assert len(pairwise_table) == 3, 'Build one row per planned pair in PAIRS.'
assert [tuple(p) for p in pairwise_table['pair']] == PAIRS, ('Use the planned directions in PAIRS, in order: '
                                                               'lower − zero, higher − zero, higher − lower.')
assert 'adjusted_p' in pairwise_table, 'Add the adjusted_p column.'
expected = pd.DataFrame([pooled_pair(group_summary, i, j, mse, df_error) for i, j in PAIRS])
assert np.allclose(pairwise_table['estimate'], expected['estimate']) and np.allclose(pairwise_table['SE'], expected['SE']), \
    'Pass group_summary, mse, and df_error from Section 3 to pooled_pair.'
assert np.allclose(pairwise_table['raw_p'], expected['raw_p']) and (pairwise_table['df'] == df_error).all(), \
    'Use the ANOVA error df (df_error) from Section 3 for every pair.'
assert np.allclose(pairwise_table['low'], expected['low']) and np.allclose(pairwise_table['high'], expected['high']), \
    "Keep pooled_pair's default alpha and family_size so each interval is a Bonferroni (98.333%) interval, not a 95% one."
assert np.allclose(pairwise_table['adjusted_p'], np.minimum(3 * expected['raw_p'], 1)), \
    'adjusted_p should be 3 × raw p, capped at 1 (np.minimum(..., 1)).'
if 'comparison' not in pairwise_table:  # re-running this cell is safe
    pairwise_table.insert(0, 'comparison', [PAIR_NAMES[tuple(p)] for p in pairwise_table['pair']])
print(f'Per-pair alpha = {family_alpha:.5f}; interval level = {1 - ALPHA / FAMILY_SIZE:.5f}; '
      f't critical = {stats.t.ppf(1 - ALPHA / (2 * FAMILY_SIZE), df_error):.4f} on {df_error} df')
display(pairwise_table.drop(columns='pair'))
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.axvspan(-MAGNITUDE_PSC, MAGNITUDE_PSC, color='#DDEBF1', label=f'±{MAGNITUDE_PSC} PSC (your planning magnitude)')
ax.axvline(0, color='black', linestyle='--', label='Zero difference')
for k, row in enumerate(pairwise_table.itertuples()):
    ax.errorbar(row.estimate, k, xerr=[[row.estimate - row.low], [row.high - row.estimate]], fmt='o', color='#13678A', capsize=6)
ax.set(yticks=range(3), yticklabels=pairwise_table.comparison, ylim=(-0.6, 2.6),
       xlabel='Difference in mean contrast (PSC) with Bonferroni interval (98.333% each)',
       title=f'{CHOICE.title()}, primary sample: three planned pairs')
ax.invert_yaxis(); ax.legend(fontsize=8, loc='lower right')
pairwise_plot = fig
fig.tight_layout(); plt.show()


### Interpret and prepare for Canvas

- Why do all three pairs share an MSE and error df, while their standard errors differ?
- Interpret one signed difference and its Bonferroni interval in PSC. What does the interval include?
- Does any pair reject at the family level (adjusted p < .05)? A raw p below .05 is not enough, as the demo's B − A showed.
- Why is the interval involving the lower-frequency and higher-frequency groups wider?

**Have ready:** locate the signed estimate, Bonferroni interval in PSC, and adjusted p for your preselected `DETAIL_PAIR`. Canvas Question 8 asks you to interpret this one pair (2–3 sentences). Notebook notes are optional.

For the Canvas reflection, Question 7 asks you to apply the shared MSE, df, and family rule to an invented comparison. Question 8 uses `pairwise_table`.


## Section 5. Does the answer hold under stricter quality?

The stricter sample repeats the **same** analysis: same outcome, groups, pairs, family, and α, with a stricter task-quality rule. It is a robustness check, not a second chance to find a difference. The two samples overlap heavily, so they are not independent replications.

**Read each interval against your planning magnitude** (separately for each sample). Under the model:

- an interval entirely inside ±`MAGNITUDE_PSC` makes differences at least that large incompatible with the data at the stated confidence level;
- an interval that crosses a boundary leaves such differences compatible. It does not show they exist.

This is interval reading, not a formal equivalence test. The table shows six readings; you'll interpret one pair. The table comparing all three magnitudes is **optional reference**; no judgments are required from it.


In [ ]:
def run_family(sample):
    # Same summaries, ANOVA, and planned family as Sections 3–4, for any sample.
    summary = sample.groupby(GROUP)[OUT].agg(n='count', mean='mean', SD='std')
    n_total = int(summary.n.sum()); dfe = n_total - 3
    m = float(((summary.n - 1) * summary.SD ** 2).sum() / dfe)
    F = stats.f_oneway(*[sample.loc[sample[GROUP].eq(k), OUT] for k in (0, 1, 2)])
    pairs = pd.DataFrame([pooled_pair(summary, i, j, m, dfe) for i, j in PAIRS])
    pairs['adjusted_p'] = np.minimum(FAMILY_SIZE * pairs.raw_p, 1)
    return summary, m, dfe, F, pairs

def magnitude_relation(lo, hi, margin):
    if lo >= margin or hi <= -margin:
        return f'entirely at/beyond ±{margin}'
    if lo > -margin and hi < margin:
        return f'entirely within ±{margin}'
    return f'crosses a ±{margin} boundary'

primary_family = run_family(primary)
assert np.allclose(primary_family[4]['estimate'], pairwise_table['estimate']), 'Section 4 must be complete before Section 5.'
stricter_family = run_family(stricter)
results_summary = pd.DataFrame([dict(sample=name, N=int(fam[0].n.sum()), F=fam[3].statistic, p=fam[3].pvalue, MSE=fam[1], df_error=fam[2])
                                for name, fam in (('primary', primary_family), ('stricter', stricter_family))])
rows = []
for name, fam in (('primary', primary_family), ('stricter', stricter_family)):
    for row in fam[4].itertuples():
        rows.append(dict(sample=name, comparison=PAIR_NAMES[tuple(row.pair)], estimate=row.estimate, low=row.low, high=row.high,
                         adjusted_p=row.adjusted_p, relation_to_your_magnitude=magnitude_relation(row.low, row.high, MAGNITUDE_PSC),
                         detail_pair=tuple(row.pair) == DETAIL_PAIR))
sensitivity_check = pd.DataFrame(rows)
reference_margins = sensitivity_check[['sample', 'comparison', 'low', 'high']].copy()
for m in (0.0133, 0.0266, 0.0399):
    reference_margins[f'±{m}'] = [magnitude_relation(lo, hi, m).replace(f' ±{m}', '').replace(' boundary', '')
                                   for lo, hi in zip(reference_margins.low, reference_margins.high)]
display(results_summary, allocation_table, sensitivity_check)
print('Optional reference (no judgments required): the same intervals against all three planning magnitudes')
display(reference_margins.drop(columns=['low', 'high']))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for offset, (name, fam, marker) in zip((-0.15, 0.15), (('primary', primary_family, 'o'), ('stricter', stricter_family, 's'))):
    for k, row in enumerate(fam[4].itertuples()):
        axes[0].errorbar(row.estimate, k + offset, xerr=[[row.estimate - row.low], [row.high - row.estimate]], fmt=marker,
                         color='#13678A' if name == 'primary' else '#B34B14', capsize=4, label=name if k == 0 else None)
axes[0].axvspan(-MAGNITUDE_PSC, MAGNITUDE_PSC, color='#DDEBF1', zorder=0)
axes[0].axvline(0, color='black', linestyle='--')
axes[0].set(yticks=range(3), yticklabels=[PAIR_NAMES[p] for p in PAIRS], xlabel='Difference (PSC), Bonferroni interval',
            title=f'Primary vs stricter; band = ±{MAGNITUDE_PSC} PSC')
axes[0].invert_yaxis(); axes[0].legend(fontsize=8)
axes[1].boxplot([primary.loc[primary[GROUP].eq(k), OUT] for k in (0, 1, 2)], positions=[0, 1, 2])
axes[1].set(xticks=[0, 1, 2], xticklabels=[LABELS[k] for k in (0, 1, 2)], ylabel='Correct stop − correct go (PSC)',
            title='People: distributions in the primary sample')
communication_figures = fig
fig.tight_layout(); plt.show()
# Optional private notes only; Canvas is the assessed response.
interpretation_record = pd.DataFrame({'comparison': [PAIR_NAMES[p] for p in PAIRS], 'primary_reading': ['', '', ''],
                                      'stricter_reading': ['', '', '']})


### Interpret and prepare for Canvas

1. **Your detail pair** (`DETAIL_PAIR`, chosen in Section 1): give its estimate and Bonferroni interval in each sample, and state what the interval says about differences of `MAGNITUDE_PSC`.
2. **Focus on your detail pair.** Did its relation to your magnitude change between samples? Small composition shifts can change an interval's reading even when no p-value changes much.
3. **Use your Section 2 plan.** What was the planning power of your detail pair for your planned difference (`allocation_plan`)? How does that inform what a nonsignificant result means for that pair? This is the power of a two-group comparison, not of the overall F-test.
4. **Communication figures.** Which of the two `communication_figures` panels best answers the question, and what does it hide?

**Have ready:** inspect the six rows of `sensitivity_check`, then interpret your preselected `DETAIL_PAIR` only. Canvas Question 9 asks for one concise sensitivity summary (2–3 sentences). Compare that pair’s primary and stricter intervals with your planned magnitude and state whether the interpretation changes. Notebook notes are optional.

For the Canvas reflection, Question 9 uses `sensitivity_check`.


## Section 6. Provenance and responsible interpretation

- **Task design.** The stop − go contrast mixes inhibition with other stimulus and response differences, and stop-signal timing adapts to each person's performance. Stricter quality cannot remove these design features. A behavioral check that would support estimating SSRT is a different question from imaging eligibility; this release has no behavioral validity flag.
- **Field documentation.** Official-style names, such as the outcome field, identify **synthetic analogs** of documented ABCD fields. `instructional_` names identify course-derived flags and groups. The release documentation, linked in Section 1, governs their meaning.
- **Sensitive reports.** Substance-use frequency groups describe reports, not diagnoses. Regional activation never justifies labeling a person or group as impaired or deficient.
- **Claims.** Same-wave, observational groups support descriptive comparisons, not causal or developmental claims. Answering those questions would need repeated visits or a different design.

For the Canvas reflection, Question 10 asks you to evaluate a claim about substance use and inhibition.


## Section 7. Finish

Complete all **three Your Turns**. Save, restart the kernel, and run all cells from the top with the same `CHOICE`, `MAGNITUDE_PSC`, and `DETAIL_PAIR`. The next cell confirms completion and keeps your plan and results in `result_record`; it does not write any files.

Then complete the **Lab 9 Canvas reflection** independently. The four written responses are bounded: one assumption check, one omnibus conclusion, one detailed pair, and one concise sensitivity summary. State your substance choice in the written questions and use your own outputs. Notebook interpretation notes are optional and are not submitted. Record AI use or non-use in Question 11. No notebook upload is required.

After the reflection closes, your instructor will release a short debrief about how this synthetic dataset was built. Keep your original answers; the debrief does not change them.


In [ ]:
completion_check = pd.DataFrame({'check': ['Three coding tasks', 'Valid planning settings', 'Primary and stricter samples', 'Six comparison rows generated'],
    'passed': [all(v is not None for v in (yt1_n, anova_result, pairwise_table)),
               MAGNITUDE_PSC in (0.0133, 0.0266, 0.0399) and DETAIL_PAIR in PAIRS,
               set(stricter.participant_id) <= set(primary.participant_id),
               len(sensitivity_check) == 6]})
assert completion_check.passed.all()
display(completion_check)
# Plan and results for your records; nothing is written to disk.
result_record = dict(notebook_version=NOTEBOOK_VERSION, dataset_version=DATASET_VERSION, csv_sha256_normalized=EXPECTED_SHA256,
                     choice=CHOICE, magnitude_PSC=MAGNITUDE_PSC, detail_pair=list(DETAIL_PAIR), alpha=ALPHA, family_size=FAMILY_SIZE,
                     python=platform.python_version(), power_plan=power_plan['plan'].astype(str).to_dict(),
                     anova=results_summary.to_dict(orient='records'), pairs=sensitivity_check.to_dict(orient='records'),
                     eligibility=eligibility_audit.to_dict(orient='records'))
print('Notebook', NOTEBOOK_VERSION, '| dataset', DATASET_VERSION, '| choice', CHOICE, '| magnitude', MAGNITUDE_PSC,
      '| results kept in result_record')


## Optional reference

The setup links the release README, data dictionary, and analysis guide. Their field definitions and eligibility rules govern this teaching file. No raw images, trial-level events, SSRT, or repeat scans are present.

For context, see the [ABCD task-fMRI documentation](https://docs.abcdstudy.org/latest/documentation/imaging/type_tfmri.html). No live lookup is needed.

**Looking ahead to Module 10:** suppose you wanted to know whether use **changes** this contrast over time, or the reverse. Which additional data and design would that need? This is optional and not assessed.
